In [1]:
import os
from pathlib import Path

DATA_ROOT = Path.home() / "mne_data"

os.environ["MNE_DATA"] = str(DATA_ROOT)
os.environ["MNE_DATASETS_BNCI_PATH"] = str(DATA_ROOT)
os.environ["MNE_DATASETS_ZHOU2016_PATH"] = str(DATA_ROOT)
os.environ["MOABB_DOWNLOAD_PROVIDER"] = "upstream"

EXPERIMENT_DIR = Path("/Users/liza/Projects/vkr/mi-ood")
CACHE_DIR = EXPERIMENT_DIR / "cache"
RESULTS_DIR = EXPERIMENT_DIR / "results"

print(EXPERIMENT_DIR)

/Users/liza/Projects/vkr/mi-ood


In [2]:
from methods import ShallowOOD
from pipelines import mi_pipelines, ood_pipelines
from processing import *

/Users/liza/Projects/bci-motor-imagery/.venv/lib/python3.11/site-packages/optuna/integration/sklearn.py:14: FutureWarning: `optuna.integration.sklearn` has been deprecated in v4.9.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.9.0. Use `optuna_integration.sklearn` instead.
  optuna_warn(f"{msg} Use `optuna_integration.sklearn` instead.", FutureWarning)
/Users/liza/Projects/bci-motor-imagery/.venv/lib/python3.11/site-packages/braindecode/models/eegpt.py:497: FutureWarning: Montage name 'standard_1020' is deprecated and will be removed in MNE 1.14. Use 'colin27_1020' instead.
  montage = mne.channels.make_standard_montage("standard_1020")
/Users/liza/Projects/bci-motor-imagery/.venv/lib/python3.11/site-packages/braindecode/models/eegpt.py:1452: FutureWarning: Montage name 'standard_1020' is deprecated and will be removed in MNE 1.14. Use 'colin27_1020' instead.
  montage = make_standard_montage("standard_1020")


In [ ]:
experiment = "within-session_7-channels"
COMMON_CHANNELS = ["FC3", "FCz", "FC4", "C3", "C4", "CP3", "CP4"]
datasets = [ "TTK", "Zhou2016", "BNCI2014_001", "Wang2025", "Yang2025_3C", "Zhou2020" ]


In [4]:
for dataset in datasets:
    print( dataset )
    subjects = get_subjects(dataset) 

    for subject in subjects:
        data = None
        for method, pipeline in mi_pipelines.items():
            result = {"dataset": dataset, "subject": subject, "experiment": experiment, "experiment_type": "mi", "method": method}
            if result_path(result).exists():
                continue
            if data is None:
                data = load_subject_data(dataset, subject, channels=COMMON_CHANNELS, within_session=True)
            train_data, val_data, test_data = data
            result["MI_BA"], result["MI_AUROC"] = MI_metrics(train_data, test_data, pipeline)
            save_result(result)
            print(f"{dataset} s{subject} {method}: {result['MI_BA']:.3f} {result['MI_AUROC']:.3f}")

        data = None
        for method, pipeline in ood_pipelines.items():
            result = {"dataset": dataset, "subject": subject, "experiment": experiment, "experiment_type": "ood", "method": method}
            if result_path(result).exists():
                continue
            if data is None:
                data = load_subject_data(dataset, subject, channels=COMMON_CHANNELS, ood_classes=("feet",), within_session=True)
                train_data, val_data, test_data = data
            model = train(train_data, pipeline)

            result["OOD_AUROC"] = OOD_AUROC(test_data, model)
            save_result(result)
            print(f"{dataset} s{subject} {method}: {result['OOD_AUROC']:.3f}")

    display_summary_by_subject(dataset, experiment, experiment_type="mi", metric="MI_BA")
    display_summary_by_subject(dataset, experiment, experiment_type="mi", metric="MI_AUROC")
    display_summary_by_subject(dataset, experiment, experiment_type="ood", metric="OOD_AUROC")

TTK


,MI_BA,n,sub-01,sub-03,sub-04,sub-05,sub-06,sub-07
method,,,,,,,,
ACM+TS+SVM,0.735000,6,0.800000,0.715000,0.760000,0.558000,0.938000,0.641000
FBCSP+ElasticNet,0.721000,6,0.767000,0.878000,0.510000,0.561000,0.929000,0.683000
Log-bandpower+shLDA,0.700000,6,0.833000,0.519000,0.715000,0.596000,0.866000,0.673000
Microstates+GB,0.646000,6,0.867000,0.519000,0.554000,0.442000,0.938000,0.554000
EEGNet,0.633000,6,0.900000,0.599000,0.558000,0.401000,0.857000,0.481000
VAR+shLDA,0.622000,6,0.733000,0.638000,0.551000,0.442000,0.804000,0.561000
ShallowConvNet,0.604000,6,0.833000,0.676000,0.478000,0.439000,0.804000,0.394000


,MI_AUROC,n,sub-01,sub-03,sub-04,sub-05,sub-06,sub-07
method,,,,,,,,
FBCSP+ElasticNet,0.804000,6,0.889000,0.962000,0.628000,0.577000,1.000000,0.769000
ACM+TS+SVM,0.773000,6,0.867000,0.724000,0.801000,0.590000,0.982000,0.673000
Log-bandpower+shLDA,0.750000,6,0.876000,0.500000,0.821000,0.628000,0.946000,0.731000
EEGNet,0.658000,6,0.991000,0.571000,0.545000,0.417000,0.964000,0.462000
VAR+shLDA,0.648000,6,0.867000,0.590000,0.603000,0.481000,0.821000,0.526000
ShallowConvNet,0.646000,6,0.960000,0.692000,0.487000,0.513000,0.768000,0.455000
Microstates+GB,0.638000,6,0.933000,0.564000,0.487000,0.397000,0.875000,0.571000


,OOD_AUROC,n,sub-01,sub-03,sub-04,sub-05,sub-06,sub-07
method,,,,,,,,
ShallowConvNet+kNN,0.762000,6,0.733000,0.775000,0.607000,0.812000,0.890000,0.753000
FBCSP+IsolationForest,0.761000,6,0.633000,0.918000,0.588000,0.664000,0.932000,0.827000
EEGNet+TempDens,0.748000,6,0.802000,0.800000,0.536000,0.596000,0.929000,0.822000
ACM+TS+OCSVM,0.688000,6,0.520000,0.757000,0.596000,0.628000,0.791000,0.837000
VAR+Mahalanobis,0.661000,6,0.676000,0.888000,0.485000,0.567000,0.606000,0.742000
Log-bandpower+Mahalanobis,0.654000,6,0.654000,0.882000,0.525000,0.626000,0.656000,0.581000
Microstates+LOF,0.583000,6,0.615000,0.502000,0.520000,0.636000,0.667000,0.559000


Zhou2016


,MI_BA,n,sub-01,sub-02,sub-03,sub-04
method,,,,,,
FBCSP+ElasticNet,0.894000,4,0.867000,0.756000,1.000000,0.955000
EEGNet,0.870000,4,0.900000,0.715000,1.000000,0.864000
ACM+TS+SVM,0.852000,4,0.733000,0.756000,0.958000,0.958000
Microstates+GB,0.841000,4,0.767000,0.718000,0.920000,0.958000
Log-bandpower+shLDA,0.825000,4,0.767000,0.753000,0.958000,0.822000
ShallowConvNet,0.773000,4,0.900000,0.718000,0.917000,0.557000
VAR+shLDA,0.685000,4,0.733000,0.635000,0.753000,0.617000


... (14 outputs left out)

Zhou2020


,MI_BA,n,sub-01,sub-02,sub-03,sub-04,sub-05,sub-06,sub-07,sub-08,sub-09,sub-10,sub-11,sub-12,sub-13,sub-14,sub-15,sub-16,sub-17,sub-18,sub-19,sub-20
method,,,,,,,,,,,,,,,,,,,,,,
ACM+TS+SVM,0.608000,20,0.633000,0.400000,0.700000,0.533000,0.600000,0.833000,0.933000,0.433000,0.467000,0.400000,0.733000,0.633000,0.533000,0.667000,0.833000,0.467000,0.459000,0.767000,0.400000,0.733000
FBCSP+ElasticNet,0.602000,20,0.500000,0.467000,0.567000,0.533000,0.533000,0.800000,0.867000,0.500000,0.733000,0.567000,0.600000,0.533000,0.467000,0.667000,0.800000,0.667000,0.515000,0.667000,0.533000,0.533000
EEGNet,0.602000,20,0.567000,0.633000,0.700000,0.400000,0.700000,0.867000,0.933000,0.500000,0.500000,0.467000,0.700000,0.667000,0.367000,0.633000,0.700000,0.500000,0.409000,0.633000,0.400000,0.767000
ShallowConvNet,0.598000,20,0.533000,0.633000,0.667000,0.367000,0.467000,0.867000,0.833000,0.533000,0.533000,0.467000,0.733000,0.667000,0.567000,0.533000,0.733000,0.467000,0.459000,0.700000,0.400000,0.800000
Log-bandpower+shLDA,0.574000,20,0.500000,0.467000,0.633000,0.367000,0.633000,0.700000,0.700000,0.700000,0.467000,0.533000,0.500000,0.567000,0.533000,0.633000,0.767000,0.533000,0.515000,0.633000,0.500000,0.600000
Microstates+GB,0.542000,20,0.600000,0.467000,0.633000,0.333000,0.600000,0.733000,0.867000,0.467000,0.467000,0.433000,0.467000,0.433000,0.400000,0.533000,0.633000,0.467000,0.512000,0.700000,0.533000,0.567000
VAR+shLDA,0.540000,20,0.433000,0.500000,0.567000,0.533000,0.467000,0.733000,0.700000,0.500000,0.633000,0.367000,0.500000,0.533000,0.400000,0.700000,0.767000,0.500000,0.408000,0.567000,0.333000,0.667000


,MI_AUROC,n,sub-01,sub-02,sub-03,sub-04,sub-05,sub-06,sub-07,sub-08,sub-09,sub-10,sub-11,sub-12,sub-13,sub-14,sub-15,sub-16,sub-17,sub-18,sub-19,sub-20
method,,,,,,,,,,,,,,,,,,,,,,
ACM+TS+SVM,0.644000,20,0.649000,0.342000,0.756000,0.636000,0.649000,0.938000,0.978000,0.440000,0.520000,0.378000,0.742000,0.640000,0.431000,0.769000,0.911000,0.556000,0.515000,0.791000,0.458000,0.787000
ShallowConvNet,0.634000,20,0.458000,0.618000,0.760000,0.382000,0.609000,0.907000,0.924000,0.498000,0.596000,0.471000,0.773000,0.582000,0.600000,0.578000,0.858000,0.640000,0.488000,0.760000,0.418000,0.769000
FBCSP+ElasticNet,0.630000,20,0.480000,0.502000,0.671000,0.640000,0.587000,0.871000,0.960000,0.569000,0.733000,0.458000,0.644000,0.604000,0.364000,0.716000,0.902000,0.653000,0.500000,0.724000,0.440000,0.582000
EEGNet,0.622000,20,0.493000,0.636000,0.787000,0.444000,0.693000,0.956000,0.982000,0.600000,0.484000,0.289000,0.818000,0.689000,0.391000,0.702000,0.840000,0.422000,0.412000,0.702000,0.378000,0.711000
Log-bandpower+shLDA,0.590000,20,0.693000,0.498000,0.609000,0.404000,0.698000,0.769000,0.787000,0.747000,0.431000,0.453000,0.538000,0.502000,0.564000,0.693000,0.791000,0.516000,0.485000,0.662000,0.413000,0.551000
Microstates+GB,0.576000,20,0.609000,0.396000,0.667000,0.493000,0.613000,0.813000,0.907000,0.462000,0.507000,0.422000,0.498000,0.436000,0.449000,0.631000,0.658000,0.507000,0.520000,0.720000,0.627000,0.582000
VAR+shLDA,0.561000,20,0.444000,0.489000,0.524000,0.529000,0.520000,0.804000,0.764000,0.596000,0.636000,0.378000,0.556000,0.613000,0.413000,0.702000,0.889000,0.502000,0.298000,0.542000,0.356000,0.658000


,OOD_AUROC,n,sub-01,sub-02,sub-03,sub-04,sub-05,sub-06,sub-07,sub-08,sub-09,sub-10,sub-11,sub-12,sub-13,sub-14,sub-15,sub-16,sub-17,sub-18,sub-19,sub-20
method,,,,,,,,,,,,,,,,,,,,,,
EEGNet+TempDens,0.642000,20,0.594000,0.567000,0.733000,0.521000,0.739000,0.879000,0.726000,0.404000,0.631000,0.561000,0.677000,0.626000,0.601000,0.738000,0.876000,0.537000,0.506000,0.587000,0.589000,0.761000
ShallowConvNet+kNN,0.636000,20,0.592000,0.598000,0.694000,0.477000,0.727000,0.809000,0.738000,0.416000,0.787000,0.488000,0.487000,0.609000,0.616000,0.752000,0.793000,0.534000,0.531000,0.704000,0.601000,0.770000
FBCSP+IsolationForest,0.629000,20,0.531000,0.631000,0.588000,0.517000,0.744000,0.787000,0.636000,0.467000,0.778000,0.499000,0.689000,0.634000,0.626000,0.658000,0.763000,0.528000,0.500000,0.851000,0.513000,0.632000
VAR+Mahalanobis,0.578000,20,0.517000,0.586000,0.598000,0.414000,0.628000,0.781000,0.588000,0.561000,0.704000,0.549000,0.618000,0.541000,0.522000,0.629000,0.609000,0.530000,0.497000,0.554000,0.484000,0.657000
Log-bandpower+Mahalanobis,0.570000,20,0.526000,0.609000,0.562000,0.458000,0.652000,0.672000,0.547000,0.417000,0.757000,0.647000,0.614000,0.558000,0.636000,0.523000,0.505000,0.528000,0.401000,0.707000,0.466000,0.608000
ACM+TS+OCSVM,0.557000,20,0.434000,0.506000,0.499000,0.440000,0.663000,0.739000,0.495000,0.495000,0.791000,0.609000,0.651000,0.544000,0.535000,0.592000,0.581000,0.537000,0.412000,0.681000,0.327000,0.603000
Microstates+LOF,0.546000,20,0.506000,0.492000,0.398000,0.468000,0.574000,0.793000,0.609000,0.509000,0.617000,0.404000,0.604000,0.573000,0.596000,0.488000,0.542000,0.531000,0.506000,0.664000,0.461000,0.576000


In [5]:
display_summary_by_dataset(
    experiment,
    mi_pipelines.keys(),
    ood_pipelines.keys(),
    experiment_specs=[
        ("MI BA", "mi", "MI_BA"),
        ("MI AUROC", "mi", "MI_AUROC"),
        ("OOD", "ood", "OOD_AUROC"),
    ],
)
